# 03 · Laboratorio de variables
Embudo del playbook: candidatas por familia → filtro rápido (Gini univariado, correlación) → filtro real (validación walk-forward `VALID_T = [7, 8, 9, 10]`, la misma de `src.train`). Referencia: `smoke_lgbm` = **0,2593**.

Los experimentos oficiales quedan en `reports/experimentos.csv` con autor `de`.

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
from src import config as C

In [2]:
from src.features import construir
from src.models import MODELOS
from src.validation import folds, metrica

def evaluar(df, feats, modelo="lgbm", seeds=(42,)):
    cats = [c for c in feats if str(df[c].dtype) == "category"]; y = df[C.TARGET]; res = []
    for s in seeds:
        g = []
        for v, itr, iva in folds(df):
            m = MODELOS[modelo](**({"seed": s} if modelo == "lgbm" else {})).fit(df.loc[itr, feats], y[itr], cats)
            g.append(metrica(y[iva], m.predict(df.loc[iva, feats])))
        res.append(g)
    return np.array(res).mean(0)

RES = []
def rep(nombre, feats, modelo="lgbm"):
    g = evaluar(df, feats, modelo); RES.append(dict(candidata=nombre, modelo=modelo, n=len(feats), gini=round(g.mean(), 4), **{f"t{v}": round(x, 4) for v, x in zip(C.VALID_T, g)}))
    print(RES[-1])

df, cols = construir(["supervivencia", "interaccion", "recencias", "reglas_negocio"])
B = cols["base"] + cols["supervivencia"] + cols["interaccion"] + cols["recencias"]
R = cols["reglas_negocio"]

## Familia 1 · Tasa histórica por segmento (target encoding temporal, R5)
Para la fila del mes `t` se usa solo la tasa de los meses `t' < t`, suavizada hacia la tasa global previa (m = 200).

In [3]:
def te_temporal(df, keys, nombre, m=200):
    """Tasa de conversión de meses ANTERIORES (t' < t) por segmento, suavizada a la tasa global previa."""
    tr = df[df.es_test == 0]
    g = tr.groupby(keys + ["t"], observed=True)[C.TARGET].agg(["sum", "count"]).reset_index()
    g = g.sort_values(keys + ["t"])
    g[["cs", "cn"]] = g.groupby(keys, observed=True)[["sum", "count"]].cumsum().values
    # estadística acumulada hasta t (inclusive) -> se usa en t+1 en adelante
    glob = tr.groupby("t")[C.TARGET].agg(["sum", "count"]).cumsum()
    out = np.full(len(df), np.nan)
    tt = df["t"].values
    for t in sorted(df.t.unique()):
        if t == 1:
            continue
        hist = g[g.t < t].groupby(keys, observed=True)[["sum", "count"]].sum()
        prior = glob.loc[:t - 1].iloc[-1]
        p0 = prior["sum"] / prior["count"]
        rate = (hist["sum"] + m * p0) / (hist["count"] + m)
        idx = np.where(tt == t)[0]
        sub = df.iloc[idx][keys]
        r = sub.merge(rate.rename("r").reset_index(), on=keys, how="left")["r"].fillna(p0).values
        out[idx] = r
    df[nombre] = out.astype("float32")
    return nombre



te = [te_temporal(df, ["banda_riesgo", "numero_productos"], "te_riesgo_prod"),
      te_temporal(df, ["banda_riesgo", "tiene_tarjeta_credito"], "te_riesgo_tarjeta"),
      te_temporal(df, ["banda_riesgo", "activo_movil"], "te_riesgo_movil"),
      te_temporal(df, ["numero_productos", "tiene_tarjeta_credito"], "te_prod_tarjeta")]
tr = df[(df.es_test == 0) & (df.t >= 2)]
print({c: round(metrica(tr.objetivo, tr[c]), 4) for c in te})
rep("baseline (smoke)", B)
rep("+ te_cruces", B + te)

{'te_riesgo_prod': np.float64(0.2153), 'te_riesgo_tarjeta': np.float64(0.1711), 'te_riesgo_movil': np.float64(0.1766), 'te_prod_tarjeta': np.float64(0.1132)}


{'candidata': 'baseline (smoke)', 'modelo': 'lgbm', 'n': 34, 'gini': np.float64(0.2593), 't7': np.float64(0.2542), 't8': np.float64(0.2621), 't9': np.float64(0.2515), 't10': np.float64(0.2695)}


{'candidata': '+ te_cruces', 'modelo': 'lgbm', 'n': 38, 'gini': np.float64(0.2578), 't7': np.float64(0.2605), 't8': np.float64(0.2573), 't9': np.float64(0.2429), 't10': np.float64(0.2704)}


**Lectura:** el Gini univariado de la tasa histórica es alto, pero **no suma** al árbol: con tan pocas categorías LightGBM ya aprende la tasa del segmento y el encoding solo agrega ruido de estimación. En el laboratorio con 3 semillas las 5 variantes (cruces, simples, recencia en tramos, hazard por duración) restaron entre 0,000 y 0,002. **Descartada.**

## Familia 2 · Relación entre columnas: la anomalía de las dos recencias

In [4]:
rep("sin familia interaccion ni dias_int", [c for c in B if c not in cols["interaccion"] + ["dias_ultima_interaccion"]])

{'candidata': 'sin familia interaccion ni dias_int', 'modelo': 'lgbm', 'n': 28, 'gini': np.float64(0.2607), 't7': np.float64(0.2654), 't8': np.float64(0.2614), 't9': np.float64(0.2469), 't10': np.float64(0.2691)}


## Familia 3 · Reglas de negocio (cruces con umbral, descubiertos en el EDA)
`alto_inactivo_180`, `bajo_3mas_prod`, `bajo_digital_tarjeta`, `rdi_alto`. Filtro rápido:

In [5]:
tr = df[df.es_test == 0]
pd.DataFrame({c: {"cobertura": tr[c].mean(), "tasa_si": tr[tr[c] == 1].objetivo.mean(), "tasa_no": tr[tr[c] == 0].objetivo.mean(),
                  "rho_max_con_base": tr[B].select_dtypes("number").corrwith(tr[c]).abs().max()} for c in R}).round(3)

,alto_inactivo_180,bajo_3mas_prod,bajo_digital_tarjeta,rdi_alto
cobertura,0.124,0.101,0.099,0.069
tasa_si,0.049,0.295,0.308,0.131
tasa_no,0.165,0.134,0.133,0.152
rho_max_con_base,0.304,0.499,0.241,0.583


In [6]:
rep("+ reglas", B + R)
rep("baseline", B, "logreg")
rep("+ reglas", B + R, "logreg")

{'candidata': '+ reglas', 'modelo': 'lgbm', 'n': 38, 'gini': np.float64(0.2604), 't7': np.float64(0.2573), 't8': np.float64(0.2713), 't9': np.float64(0.2429), 't10': np.float64(0.2702)}


{'candidata': 'baseline', 'modelo': 'logreg', 'n': 34, 'gini': np.float64(0.2408), 't7': np.float64(0.2441), 't8': np.float64(0.2485), 't9': np.float64(0.2305), 't10': np.float64(0.2401)}


{'candidata': '+ reglas', 'modelo': 'logreg', 'n': 38, 'gini': np.float64(0.2624), 't7': np.float64(0.2674), 't8': np.float64(0.2754), 't9': np.float64(0.2504), 't10': np.float64(0.2563)}


**Lectura:** el árbol no gana (ya encuentra los umbrales), pero la **logística sube +0,02** y pasa al árbol: la señal es casi aditiva y solo le faltaban estos 4 quiebres.

## Familia 4 · Exclusión de ruido (`sin_ruido`)
Quita 7 columnas originales sin señal. Se implementa como familia de exclusión (`'-col'`).

In [7]:
df2, cols2 = construir(["supervivencia", "reglas_negocio", "sin_ruido"])
X = cols2["base"] + cols2["reglas_negocio"]
df = df2
rep("reglas + sin_ruido", X, "logreg")
rep("reglas + sin_ruido + supervivencia", X + cols2["supervivencia"], "logreg")
rep("reglas + sin_ruido", X, "lgbm")

{'candidata': 'reglas + sin_ruido', 'modelo': 'logreg', 'n': 19, 'gini': np.float64(0.2677), 't7': np.float64(0.275), 't8': np.float64(0.2759), 't9': np.float64(0.2581), 't10': np.float64(0.2618)}


{'candidata': 'reglas + sin_ruido + supervivencia', 'modelo': 'logreg', 'n': 23, 'gini': np.float64(0.2655), 't7': np.float64(0.2721), 't8': np.float64(0.275), 't9': np.float64(0.2538), 't10': np.float64(0.2611)}


{'candidata': 'reglas + sin_ruido', 'modelo': 'lgbm', 'n': 19, 'gini': np.float64(0.2606), 't7': np.float64(0.263), 't8': np.float64(0.2748), 't9': np.float64(0.2418), 't10': np.float64(0.2629)}


## Resumen y ensamble (experimentos oficiales)

In [8]:
tab = pd.DataFrame(RES); display(tab)
log = pd.read_csv("reports/experimentos.csv")
log[log.nombre.isin(["smoke_lgbm", "de_logreg_ref", "de_logreg_reglas", "de_logreg_reglas_sinruido", "de_lgbm_reglas",
                     "de_lgbm_sinruido", "de_lgbm_reglas_sinruido", "de_ens_logreg_lgbm"])][
    ["nombre", "modelo", "familias", "gini_t7", "gini_t8", "gini_t9", "gini_t10", "gini_medio", "gini_std"]]

,candidata,modelo,n,gini,t7,t8,t9,t10
0,baseline (smoke),lgbm,34,0.2593,0.2542,0.2621,0.2515,0.2695
1,+ te_cruces,lgbm,38,0.2578,0.2605,0.2573,0.2429,0.2704
2,sin familia interaccion ni dias_int,lgbm,28,0.2607,0.2654,0.2614,0.2469,0.2691
3,+ reglas,lgbm,38,0.2604,0.2573,0.2713,0.2429,0.2702
4,baseline,logreg,34,0.2408,0.2441,0.2485,0.2305,0.2401
5,+ reglas,logreg,38,0.2624,0.2674,0.2754,0.2504,0.2563
6,reglas + sin_ruido,logreg,19,0.2677,0.2750,0.2759,0.2581,0.2618
7,reglas + sin_ruido + supervivencia,logreg,23,0.2655,0.2721,0.2750,0.2538,0.2611
8,reglas + sin_ruido,lgbm,19,0.2606,0.2630,0.2748,0.2418,0.2629


,nombre,modelo,familias,gini_t7,gini_t8,gini_t9,gini_t10,gini_medio,gini_std
0,smoke_lgbm,lgbm,supervivencia+interaccion+recencias,0.2542,0.2621,0.2515,0.2695,0.2593,0.0070
6,de_logreg_ref,logreg,supervivencia+interaccion+recencias,0.2441,0.2485,0.2305,0.2401,0.2408,0.0067
7,de_logreg_reglas,logreg,supervivencia+interaccion+recencias+reglas_neg...,0.2674,0.2754,0.2504,0.2563,0.2624,0.0097
8,de_logreg_reglas_sinruido,logreg,reglas_negocio+sin_ruido,0.2750,0.2759,0.2581,0.2618,0.2677,0.0079
10,de_lgbm_reglas,lgbm,supervivencia+interaccion+recencias+reglas_neg...,0.2573,0.2713,0.2429,0.2702,0.2604,0.0115
12,de_lgbm_sinruido,lgbm,supervivencia+interaccion+recencias+sin_ruido,0.2630,0.2657,0.2479,0.2741,0.2627,0.0095
14,de_lgbm_reglas_sinruido,lgbm,reglas_negocio+sin_ruido,0.2630,0.2748,0.2418,0.2629,0.2606,0.0119
16,de_ens_logreg_lgbm,rank_avg,de_logreg_reglas_sinruido+de_lgbm_sinruido,0.2743,0.2748,0.2563,0.2706,0.2690,0.0075


## Ganadoras
| Familia | Uso recomendado | Gini medio | Delta |
|---|---|---|---|
| `reglas_negocio` + `sin_ruido` | logreg | 0,2677 | +0,0269 vs logreg ref (4/4 folds) |
| `sin_ruido` | lgbm con supervivencia, interaccion, recencias | 0,2627 | +0,0034 vs smoke (3/4) |
| Ensamble de ambos (rank avg) | — | **0,2690** | **+0,0097 vs smoke (4/4)** |

Descartadas: target encoding temporal (−0,000 a −0,002), cruces numéricos (−0,000), logs (−0,001 en logreg), supervivencia en logreg (−0,002).

## ¿Qué significa para el banco?
Un modelo **simple y explicable** (logística con 4 reglas de negocio) ordena mejor que el boosting. Cada regla se lee en una frase: *riesgo alto inactivo 6 meses no convierte*, *riesgo bajo con 3+ productos sí*, *el digital con tarjeta aún más*, *el sobreendeudado menos*. Es fácil de defender ante el comité de riesgos y ante la SBS, y de recalibrar cada mes.